# Pipeline final — clusters × agentes sintéticos

Notebook enxuto para executar apenas o pipeline final do projeto:

```text
parquet de clusters
cluster_id | resumo | tema | macrotema | motivo
↓
base de agentes 1000
↓
amostra estratificada padrão 200
↓
agente sintético × cluster
↓
checkpoints parquet
↓
arquivo completo + agregados
```

Este notebook **não usa base de notícias/publicações**. O único input do estímulo é o arquivo parquet de clusters.

## 0. Imports

In [1]:
# Rode uma vez, se necessário:
# !pip install -U pandas numpy pydantic langchain langchain-openai tqdm pyarrow

from __future__ import annotations

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Optional
import math
import os
import sys
import time
import warnings

import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None

from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

warnings.filterwarnings("ignore", category=UserWarning, module="pydantic")

pd.set_option("display.max_columns", 180)
pd.set_option("display.max_colwidth", 220)

## 1. Configuração

Ajuste principalmente:

- `PROJECT_DIR`
- `CLUSTERS_PARQUET`
- `REFERENCE_CONFIG_PATH`, se for gerar agentes do zero
- `AGENTS_EXISTING_PATH`, caso já exista uma base de agentes pronta

In [2]:
# ============================================================
# AJUSTE AQUI
# ============================================================

PROJECT_DIR = Path(r"C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm")

# Arquivo de clusters em parquet.
# Estrutura esperada:
# Index(['cluster_id', 'resumo', 'tema', 'macrotema', 'motivo'], dtype='object')
CLUSTERS_PARQUET = Path(
     r"C:\Users\claudio.bruno_cortex\Documents\código_python\research\data\alimentos_e_bebidas\cluster_temas_ambev.parquet"
)
# Se você já tiver uma base de agentes pronta, aponte aqui.
# Pode ser .csv, .parquet ou .xlsx. Se None, o notebook tenta gerar com generate_profiles().
AGENTS_EXISTING_PATH = None

DATA_OUTPUT_DIR = PROJECT_DIR / "data" / "output"
ABERJE_OUTPUT_DIR = PROJECT_DIR / "data" / "aberje_outputs"
REFERENCE_CONFIG_PATH = PROJECT_DIR / "data" / "reference" / "reference_config.yml"

RUN_NAME = "pipeline_final_clusters_x_agents_n1000_sample200_seed42"
RUN_OUTPUT_DIR = ABERJE_OUTPUT_DIR / RUN_NAME

MODEL = "gpt-4o-mini"
TEMPERATURE = 0.2

N_AGENTS_POPULATION = 1_000
N_AGENTS_SAMPLE = 200
RANDOM_STATE = 42

MAX_WORKERS = 30
CHECKPOINT_SIZE = 100
MAX_RETRIES = 2
RETRY_SLEEP = 2.0

# Use valores pequenos para teste, por exemplo 5 agentes e 3 clusters.
# Para rodar completo, deixe como None.
LIMIT_AGENTS = None
LIMIT_CLUSTERS = None

# Se True, força nova geração da população de agentes mesmo se já existir CSV salvo.
FORCE_GENERATE_AGENTS = False

# Se True, remove checkpoints e saídas anteriores do mesmo RUN_NAME antes de rodar.
CLEAR_EXISTING_OUTPUTS = True

# Controle geral da execução.
RUN_PIPELINE = True

RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Pasta do run:", RUN_OUTPUT_DIR)

Pasta do run: C:\Users\claudio.bruno_cortex\Documents\código_python\synthetic-audiences-abm\data\aberje_outputs\pipeline_final_clusters_x_agents_n1000_sample200_seed42


## 2. Colunas esperadas, constructos e questionário

In [3]:
AGENT_ID_COLUMN = "id_persona"

PERSONA_COLUMNS = [
    "Persona",
]

SOCIO_DEMOGRAPHIC_COLUMNS = [
    "idade",
    "gênero",
    "escolaridade",
    "situação ocupacional",
    "estado civil",
    "condição de moradia",
    "tipo de moradia",
    "localização",
    "renda familiar mensal",
    "acesso à internet",
    "tamanho do domicílio",
    "telefone",
]

VALUES_ATTITUDES_PERCEPTIONS_COLUMNS = [
    "identificação política",
    "ideologia",
    "raça/cor",
    "geração",
    "segmento_e_bolhas",
    "tipo_de_território",
    "uf",
    "tipo_de_fontes_de_informação",
    "Religiosidade",
    "Família",
    "Honestidade",
    "Meritocracia",
    "Ideologia",
    "Discriminação",
    "Sexualidade",
    "Comportamento_social",
    "Identidade_nacional",
    "Insegurança",
    "Confiança",
]

AGENT_CONTEXT_COLUMNS = (
    [AGENT_ID_COLUMN]
    + PERSONA_COLUMNS
    + SOCIO_DEMOGRAPHIC_COLUMNS
    + VALUES_ATTITUDES_PERCEPTIONS_COLUMNS
)

CLUSTER_COLUMNS = ["cluster_id", "resumo", "tema", "macrotema", "motivo"]

ITEM_COLUMNS = [
    "VAL_01", "VAL_02", "VAL_03", "VAL_04",
    "POS_01", "POS_02", "POS_03", "POS_04", "POS_05",
    "CRE_01", "CRE_02", "CRE_03",
]

CONSTRUCT_MAP = {
    "Valencia": ["VAL_01", "VAL_02", "VAL_03", "VAL_04"],
    "Emocao_Positiva": ["POS_01", "POS_02", "POS_03", "POS_04", "POS_05"],
    "Credibilidade": ["CRE_01", "CRE_02", "CRE_03"],
}


class SurveyResponse(BaseModel):
    """Schema estruturado do questionário usado pelos agentes sintéticos."""

    # VALÊNCIA
    VAL_01: int = Field(ge=1, le=7)
    VAL_02: int = Field(ge=1, le=7)
    VAL_03: int = Field(ge=1, le=7)
    VAL_04: int = Field(ge=1, le=7)

    # EMOÇÃO POSITIVA
    POS_01: int = Field(ge=1, le=7)
    POS_02: int = Field(ge=1, le=7)
    POS_03: int = Field(ge=1, le=7)
    POS_04: int = Field(ge=1, le=7)
    POS_05: int = Field(ge=1, le=7)

    # CREDIBILIDADE
    CRE_01: int = Field(ge=1, le=7)
    CRE_02: int = Field(ge=1, le=7)
    CRE_03: int = Field(ge=1, le=7)


def build_questionnaire_text() -> str:
    """
    Mantém o mesmo questionário reduzido usado no notebook do projeto.

    Escala de todos os itens: inteiros de 1 a 7.
    """
    return """
QUESTIONÁRIO

Responda todos os itens usando apenas números inteiros de 1 a 7.

VALÊNCIA
Após ler o conteúdo acima, minha reação foi...

VAL_01: 1 = infeliz | 7 = feliz
VAL_02: 1 = desagradado(a) | 7 = agradado(a)
VAL_03: 1 = insatisfeito(a) | 7 = satisfeito(a)
VAL_04: 1 = melancólico(a) | 7 = contente

EMOÇÃO POSITIVA
Ao ler o conteúdo acima, senti-me...
Escala: 1 = nada | 4 = moderadamente | 7 = extremamente.

POS_01: alerta
POS_02: inspirado(a)
POS_03: determinado(a)
POS_04: atento(a)
POS_05: ativo(a)

CREDIBILIDADE DA MENSAGEM
O quanto cada adjetivo descreve o conteúdo acima?
Escala: 1 = descreve muito mal | 4 = intermediário | 7 = descreve muito bem.

CRE_01: preciso
CRE_02: autêntico
CRE_03: crível
""".strip()

print(build_questionnaire_text())

QUESTIONÁRIO

Responda todos os itens usando apenas números inteiros de 1 a 7.

VALÊNCIA
Após ler o conteúdo acima, minha reação foi...

VAL_01: 1 = infeliz | 7 = feliz
VAL_02: 1 = desagradado(a) | 7 = agradado(a)
VAL_03: 1 = insatisfeito(a) | 7 = satisfeito(a)
VAL_04: 1 = melancólico(a) | 7 = contente

EMOÇÃO POSITIVA
Ao ler o conteúdo acima, senti-me...
Escala: 1 = nada | 4 = moderadamente | 7 = extremamente.

POS_01: alerta
POS_02: inspirado(a)
POS_03: determinado(a)
POS_04: atento(a)
POS_05: ativo(a)

CREDIBILIDADE DA MENSAGEM
O quanto cada adjetivo descreve o conteúdo acima?
Escala: 1 = descreve muito mal | 4 = intermediário | 7 = descreve muito bem.

CRE_01: preciso
CRE_02: autêntico
CRE_03: crível


## 3. Funções utilitárias e prompt

In [4]:
def safe_get(row: pd.Series, column: str, default: str = "Não informado") -> str:
    """Busca uma coluna em uma linha do DataFrame, tratando ausência e valores nulos."""
    if column not in row.index:
        return default
    value = row[column]
    if pd.isna(value):
        return default
    return str(value)


def validate_columns(
    df: pd.DataFrame,
    required_columns: list[str],
    name: str,
    raise_error: bool = True,
) -> list[str]:
    """Valida se um DataFrame contém as colunas necessárias para o pipeline."""
    missing = [col for col in required_columns if col not in df.columns]
    if missing and raise_error:
        raise ValueError(f"{name} não contém todas as colunas esperadas. Colunas ausentes: {missing}")
    return missing


def format_column_block(row: pd.Series, columns: list[str]) -> str:
    """Formata um grupo de colunas como bloco textual para inserir no prompt."""
    return "\n".join(f"- {col}: {safe_get(row, col)}" for col in columns)


def truncate_text(value: str, max_chars: int = 5_000) -> str:
    """Limita textos longos para controlar custo, latência e risco de estourar contexto."""
    value = "" if pd.isna(value) else str(value)
    if len(value) <= max_chars:
        return value
    return value[:max_chars].rsplit(" ", 1)[0] + " [...]"


def build_agent_context(agent_row: pd.Series) -> str:
    """Constrói o contexto completo do agente a partir da linha da base de personas."""
    persona = safe_get(agent_row, "Persona")
    socio = format_column_block(agent_row, SOCIO_DEMOGRAPHIC_COLUMNS)
    values = format_column_block(agent_row, VALUES_ATTITUDES_PERCEPTIONS_COLUMNS)

    return f"""
PERSONA

A variável Persona é a descrição narrativa principal do agente.
Ela sintetiza quem é essa pessoa, seu contexto social, seus hábitos, tensões e visão geral de mundo.
Use a Persona como primeiro guia de interpretação.

{persona}

VARIÁVEIS SOCIO-DEMOGRÁFICAS

Essas variáveis descrevem a posição social objetiva do agente:
idade, gênero, escolaridade, ocupação, estado civil, moradia, localização, renda,
acesso à internet, tamanho do domicílio e telefone.
Elas devem orientar como o agente interpreta riscos, oportunidades, empresas,
instituições, notícias e temas públicos.

{socio}

VALORES, ATITUDES E PERCEPÇÕES

Essas variáveis descrevem a visão de mundo do agente:
identificação política, ideologia, raça/cor, geração, segmento sociopolítico,
território, UF, fontes de informação e dimensões psicográficas.
Elas devem orientar a resposta do agente em termos de confiança, insegurança,
moralidade, religião, família, meritocracia, discriminação, sexualidade,
identidade nacional e comportamento social.

{values}
""".strip()


def build_cluster_context(cluster_row: pd.Series, max_chars: int = 5_000) -> str:
    """Constrói o estímulo textual a partir do parquet de clusters, sem usar base de notícias."""
    resumo = truncate_text(safe_get(cluster_row, "resumo"), max_chars=max_chars)
    motivo = truncate_text(safe_get(cluster_row, "motivo"), max_chars=1_500)

    return f"""
CLUSTER / ESTÍMULO SINTÉTICO

Cluster ID: {safe_get(cluster_row, "cluster_id")}
Tema: {safe_get(cluster_row, "tema")}
Macrotema: {safe_get(cluster_row, "macrotema")}

Resumo do cluster:
{resumo}

Motivo da classificação do macrotema:
{motivo}
""".strip()


def get_agent_metadata(agent_row: pd.Series) -> dict:
    """Retorna os metadados do agente que devem acompanhar cada resposta."""
    metadata_cols = [
        "id_persona",
        "idade",
        "gênero",
        "escolaridade",
        "situação ocupacional",
        "estado civil",
        "condição de moradia",
        "tipo de moradia",
        "localização",
        "renda familiar mensal",
        "acesso à internet",
        "tamanho do domicílio",
        "telefone",
        "identificação política",
        "ideologia",
        "raça/cor",
        "geração",
        "segmento_e_bolhas",
        "tipo_de_território",
        "uf",
        "tipo_de_fontes_de_informação",
    ]
    return {col: safe_get(agent_row, col) for col in metadata_cols if col in agent_row.index}


def get_cluster_metadata(cluster_row: pd.Series) -> dict:
    """Retorna os metadados do cluster que devem acompanhar cada resposta."""
    return {
        "cluster_id": safe_get(cluster_row, "cluster_id"),
        "cluster_resumo": safe_get(cluster_row, "resumo"),
        "cluster_tema": safe_get(cluster_row, "tema"),
        "cluster_macrotema": safe_get(cluster_row, "macrotema"),
        "cluster_motivo": safe_get(cluster_row, "motivo"),
    }


def build_prompt(agent_row: pd.Series, cluster_row: pd.Series) -> str:
    """Monta o prompt enviado ao modelo para cada par agente × cluster."""
    questionnaire_text = build_questionnaire_text()
    agent_context = build_agent_context(agent_row)
    cluster_context = build_cluster_context(cluster_row)

    return f"""
Você é a pessoa descrita abaixo.

Sua tarefa é responder a um questionário de pesquisa como se fosse essa pessoa real.
Não responda como especialista, analista, consultor ou modelo de IA.
Responda a partir da Persona, das variáveis socio-demográficas e dos valores, atitudes e percepções.

CONTEXTO DO AGENTE

{agent_context}

{cluster_context}

INSTRUÇÕES SOBRE O ESTÍMULO

- O estímulo acima é um cluster sintético, não uma notícia individual.
- Não há veículo, jornalista, link ou fonte específica nesta etapa.
- Responda considerando o resumo, o tema, o macrotema e o motivo da classificação.
- Avalie como esse conteúdo consolidado tenderia a ser recebido pelo agente.

INSTRUÇÕES GERAIS DE RESPOSTA

- Você foi exposto ao conteúdo acima.
- Responda todos os itens do questionário.
- Use apenas números inteiros entre 1 e 7.
- Não inclua justificativas.
- Não inclua texto explicativo.
- Não inclua comentários.
- Retorne somente um JSON válido no schema solicitado.
- Se algum item parecer menos aplicável, ainda assim responda pela impressão geral do agente.

{questionnaire_text}
""".strip()

## 4. Configurar LLM estruturado

In [5]:
if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError(
        "OPENAI_API_KEY não encontrada. Configure a variável de ambiente antes de rodar."
    )

llm = ChatOpenAI(
    model=MODEL,
    temperature=TEMPERATURE,
)

structured_llm = llm.with_structured_output(SurveyResponse)

print("LLM estruturado configurado.")

LLM estruturado configurado.


## 5. Gerar/carregar 1000 agentes e criar amostra padrão de 200

In [6]:
def import_generate_profiles():
    """
    Tenta localizar a função generate_profiles no projeto local.

    Ajuste a lista import_attempts se a função estiver em outro módulo.
    """
    project_str = str(PROJECT_DIR)
    if project_str not in sys.path:
        sys.path.append(project_str)

    import_attempts = [
        "generate_profiles",
        "src.generate_profiles",
        "src.agents.generate_profiles",
        "src.population.generate_profiles",
        "synthetic_audiences.generate_profiles",
        "synthetic_audiences.agents.generate_profiles",
    ]

    last_error = None

    for module_name in import_attempts:
        try:
            module = __import__(module_name, fromlist=["generate_profiles"])
            return getattr(module, "generate_profiles")
        except Exception as exc:
            last_error = exc

    raise ImportError(
        "Não encontrei generate_profiles automaticamente. "
        "Ajuste import_attempts dentro de import_generate_profiles() "
        "ou defina AGENTS_EXISTING_PATH com uma base de agentes já salva. "
        f"Último erro: {repr(last_error)}"
    )


def read_dataframe(path: str | Path) -> pd.DataFrame:
    """Lê DataFrame a partir de .parquet, .csv ou .xlsx."""
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {path}")

    suffix = path.suffix.lower()

    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path)

    raise ValueError(f"Formato não suportado: {suffix}")


def load_or_generate_agents(
    n_agents: int = N_AGENTS_POPULATION,
    seed: int = RANDOM_STATE,
    force_generate: bool = FORCE_GENERATE_AGENTS,
) -> pd.DataFrame:
    """
    Carrega a população de agentes, se existir; caso contrário, gera uma nova.

    Prioridade:
    1. AGENTS_EXISTING_PATH, se informado;
    2. CSV padrão salvo em DATA_OUTPUT_DIR;
    3. geração via generate_profiles().
    """
    if AGENTS_EXISTING_PATH is not None:
        df_agents = read_dataframe(AGENTS_EXISTING_PATH)
        validate_columns(df_agents, AGENT_CONTEXT_COLUMNS, "df_agents", raise_error=True)
        print(f"População carregada de AGENTS_EXISTING_PATH: {AGENTS_EXISTING_PATH} | linhas: {len(df_agents)}")
        return df_agents

    agents_csv = DATA_OUTPUT_DIR / f"agents_population_n{n_agents}_seed{seed}.csv"
    agents_csv_ptbr = DATA_OUTPUT_DIR / f"agents_population_n{n_agents}_seed{seed}_ptbr_semicolon.csv"

    if agents_csv.exists() and not force_generate:
        df_agents = pd.read_csv(agents_csv)
        validate_columns(df_agents, AGENT_CONTEXT_COLUMNS, "df_agents", raise_error=True)
        print(f"População carregada: {agents_csv} | linhas: {len(df_agents)}")
        return df_agents

    generate_profiles = import_generate_profiles()

    df_agents = generate_profiles(
        n=n_agents,
        output_csv=agents_csv,
        config_path=str(REFERENCE_CONFIG_PATH),
        seed=seed,
    )

    validate_columns(df_agents, AGENT_CONTEXT_COLUMNS, "df_agents", raise_error=True)

    df_agents.to_csv(agents_csv, index=False, encoding="utf-8-sig")
    df_agents.to_csv(agents_csv_ptbr, index=False, sep=";", encoding="utf-8-sig")

    print(f"População gerada: {agents_csv} | linhas: {len(df_agents)}")
    print(f"Versão PT-BR salva: {agents_csv_ptbr}")

    return df_agents


def make_agent_sample(
    df_agents: pd.DataFrame,
    target: int = N_AGENTS_SAMPLE,
    stratify_col: str = "segmento_e_bolhas",
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    """
    Cria amostra estratificada de agentes.

    Regra:
    - preserva a distribuição proporcional de segmento_e_bolhas;
    - ajusta arredondamentos para chegar ao target;
    - nunca sorteia mais agentes do que existem em cada segmento.
    """
    validate_columns(df_agents, AGENT_CONTEXT_COLUMNS, "df_agents", raise_error=True)

    if stratify_col not in df_agents.columns:
        raise ValueError(f"Coluna de estratificação ausente: {stratify_col}")

    if target > len(df_agents):
        raise ValueError(f"O target ({target}) é maior que o total disponível ({len(df_agents)}).")

    counts = df_agents[stratify_col].value_counts(dropna=False)
    proportions = counts / counts.sum()

    allocation = pd.DataFrame({
        stratify_col: counts.index,
        "available": counts.values,
        "raw_n": [target * proportions.loc[idx] for idx in counts.index],
    })

    allocation["n"] = allocation["raw_n"].apply(math.floor).astype(int)
    allocation["remainder"] = allocation["raw_n"] - allocation["n"]

    # Garante ao menos 1 agente por segmento quando o target comporta isso.
    if target >= len(allocation):
        allocation.loc[allocation["n"] == 0, "n"] = 1

    # Corrige eventual excesso gerado pela regra de mínimo 1.
    while allocation["n"].sum() > target:
        candidates = allocation[allocation["n"] > 1].sort_values("remainder", ascending=True)
        idx = candidates.index[0]
        allocation.loc[idx, "n"] -= 1

    # Distribui vagas restantes pelos maiores restos.
    while allocation["n"].sum() < target:
        candidates = allocation[allocation["n"] < allocation["available"]].sort_values(
            "remainder",
            ascending=False,
        )
        if candidates.empty:
            break
        idx = candidates.index[0]
        allocation.loc[idx, "n"] += 1

    samples = []

    for _, row in allocation.iterrows():
        segment = row[stratify_col]
        n = int(min(row["n"], row["available"]))
        df_segment = df_agents[df_agents[stratify_col].eq(segment)]

        samples.append(
            df_segment.sample(
                n=n,
                random_state=random_state,
                replace=False,
            )
        )

    df_sample = (
        pd.concat(samples, ignore_index=True)
        .sample(frac=1, random_state=random_state)
        .reset_index(drop=True)
    )

    sample_path = RUN_OUTPUT_DIR / f"agents_sample_n{len(df_sample)}_seed{random_state}.csv"
    df_sample.to_csv(sample_path, index=False, sep=";", encoding="utf-8-sig")

    print("Amostra salva em:", sample_path)
    print("Total amostra:", len(df_sample))
    display(df_sample[stratify_col].value_counts().to_frame("n_agentes"))

    return df_sample

## 6. Carregar parquet de clusters

In [7]:
def load_clusters_from_parquet(path: str | Path = CLUSTERS_PARQUET) -> pd.DataFrame:
    """
    Carrega o parquet de clusters.

    Estrutura esperada:
    - cluster_id
    - resumo
    - tema
    - macrotema
    - motivo
    """
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Arquivo de clusters não encontrado: {path}")

    df_clusters = pd.read_parquet(path).copy()
    validate_columns(df_clusters, CLUSTER_COLUMNS, "df_clusters", raise_error=True)

    # Mantém apenas as colunas necessárias e padroniza tipos textuais.
    df_clusters = df_clusters[CLUSTER_COLUMNS].copy()

    for col in CLUSTER_COLUMNS:
        df_clusters[col] = df_clusters[col].astype(str)

    # Remove linhas sem resumo, pois o estímulo precisa de conteúdo mínimo.
    df_clusters = df_clusters[df_clusters["resumo"].str.strip().ne("")].reset_index(drop=True)

    if df_clusters.empty:
        raise ValueError("O parquet de clusters foi carregado, mas ficou vazio após remover resumos em branco.")

    print("Clusters carregados:", len(df_clusters))
    print("Colunas:", list(df_clusters.columns))
    display(df_clusters.head())

    return df_clusters

## 7. Rodar agentes sintéticos sobre clusters

In [8]:
def model_response_to_dict(response) -> dict:
    """Converte resposta Pydantic v1/v2 para dict."""
    if hasattr(response, "model_dump"):
        return response.model_dump()
    if hasattr(response, "dict"):
        return response.dict()
    return dict(response)


def empty_item_response() -> dict:
    """Cria resposta vazia para linhas com erro, preservando colunas do questionário."""
    return {col: np.nan for col in ITEM_COLUMNS}


def evaluate_cluster_for_agent(
    agent_row: pd.Series,
    cluster_row: pd.Series,
    structured_llm,
    model: str = MODEL,
    max_retries: int = MAX_RETRIES,
    retry_sleep: float = RETRY_SLEEP,
) -> dict:
    """
    Avalia um cluster para um agente sintético.

    Cada chamada retorna:
    - respostas 1-7 do questionário;
    - metadados do agente;
    - metadados do cluster.
    """
    prompt = build_prompt(agent_row=agent_row, cluster_row=cluster_row)
    last_error = None

    for attempt in range(max_retries + 1):
        try:
            response = structured_llm.invoke(prompt)
            result = model_response_to_dict(response)
            result.update(get_agent_metadata(agent_row))
            result.update(get_cluster_metadata(cluster_row))
            result.update({
                "stimulus_type": "cluster_sintetico",
                "model": model,
                "error": "",
            })
            return result

        except Exception as exc:
            last_error = repr(exc)
            if attempt < max_retries:
                time.sleep(retry_sleep * (attempt + 1))

    # Resultado de erro preserva metadados para diagnóstico e reprocessamento.
    error_result = empty_item_response()
    error_result.update(get_agent_metadata(agent_row))
    error_result.update(get_cluster_metadata(cluster_row))
    error_result.update({
        "stimulus_type": "cluster_sintetico",
        "model": model,
        "error": last_error,
    })
    return error_result


def iter_cluster_task_batches(
    df_agents_run: pd.DataFrame,
    df_clusters_run: pd.DataFrame,
    batch_size: int = CHECKPOINT_SIZE,
):
    """Gera lotes agente × cluster sem criar uma lista gigante em memória."""
    batch = []

    for cluster_idx in df_clusters_run.index:
        for agent_idx in df_agents_run.index:
            batch.append((agent_idx, cluster_idx))

            if len(batch) >= batch_size:
                yield batch
                batch = []

    if batch:
        yield batch


def clear_previous_outputs(output_dir: str | Path = RUN_OUTPUT_DIR) -> None:
    """Remove checkpoints e saídas finais do run atual para evitar duplicação ao reexecutar."""
    output_dir = Path(output_dir)

    patterns = [
        "cluster_responses_checkpoint_*.parquet",
        "manifest_cluster_checkpoints.csv",
        "synthetic_clusters_complete.parquet",
        "synthetic_clusters_complete.csv",
    ]

    for pattern in patterns:
        for path in output_dir.glob(pattern):
            path.unlink(missing_ok=True)

    agg_dir = output_dir / "aggregations"
    if agg_dir.exists():
        for path in agg_dir.glob("*.parquet"):
            path.unlink(missing_ok=True)
        for path in agg_dir.glob("*.csv"):
            path.unlink(missing_ok=True)


def run_synthetic_audience_clusters(
    df_agents: pd.DataFrame,
    df_clusters: pd.DataFrame,
    output_dir: str | Path = RUN_OUTPUT_DIR,
    max_workers: int = MAX_WORKERS,
    checkpoint_size: int = CHECKPOINT_SIZE,
    max_retries: int = MAX_RETRIES,
    retry_sleep: float = RETRY_SLEEP,
    limit_agents: Optional[int] = LIMIT_AGENTS,
    limit_clusters: Optional[int] = LIMIT_CLUSTERS,
) -> None:
    """
    Executa agentes sintéticos sobre clusters.

    Cada linha de saída representa 1 agente × 1 cluster.
    """
    validate_columns(df_agents, AGENT_CONTEXT_COLUMNS, "df_agents", raise_error=True)
    validate_columns(df_clusters, CLUSTER_COLUMNS, "df_clusters", raise_error=True)

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    df_agents_run = df_agents.copy()
    df_clusters_run = df_clusters.copy()

    if limit_agents is not None:
        df_agents_run = df_agents_run.head(limit_agents)

    if limit_clusters is not None:
        df_clusters_run = df_clusters_run.head(limit_clusters)

    total_tasks = len(df_agents_run) * len(df_clusters_run)

    print("Agentes no run:", len(df_agents_run))
    print("Clusters no run:", len(df_clusters_run))
    print("Total agente × cluster:", total_tasks)
    print("Max workers:", max_workers)
    print("Checkpoint size:", checkpoint_size)

    if total_tasks == 0:
        print("Nenhuma tarefa para executar.")
        return None

    checkpoint_paths = []
    processed = 0
    batch_id = 0

    for batch in iter_cluster_task_batches(
        df_agents_run=df_agents_run,
        df_clusters_run=df_clusters_run,
        batch_size=checkpoint_size,
    ):
        batch_id += 1
        batch_results = []

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_task = {
                executor.submit(
                    evaluate_cluster_for_agent,
                    df_agents_run.loc[agent_idx],
                    df_clusters_run.loc[cluster_idx],
                    structured_llm,
                    MODEL,
                    max_retries,
                    retry_sleep,
                ): (agent_idx, cluster_idx)
                for agent_idx, cluster_idx in batch
            }

            completed = as_completed(future_to_task)

            if tqdm is not None:
                completed = tqdm(
                    completed,
                    total=len(future_to_task),
                    desc=f"Batch {batch_id} | {processed}/{total_tasks}",
                )

            for future in completed:
                batch_results.append(future.result())

        df_batch = pd.DataFrame(batch_results)
        checkpoint_path = output_dir / f"cluster_responses_checkpoint_{batch_id:05d}.parquet"
        df_batch.to_parquet(checkpoint_path, index=False)

        checkpoint_paths.append(checkpoint_path)
        processed += len(df_batch)

        print(
            f"Checkpoint salvo: {checkpoint_path.name} | "
            f"Processados: {processed}/{total_tasks}"
        )

    manifest = pd.DataFrame({
        "checkpoint_path": [str(path) for path in checkpoint_paths],
    })

    manifest_path = output_dir / "manifest_cluster_checkpoints.csv"
    manifest.to_csv(manifest_path, index=False, sep=";", encoding="utf-8-sig")

    print("Manifest salvo em:", manifest_path)
    print("Run finalizado.")

## 8. Consolidar completo e gerar agregados

In [9]:
def compute_construct_scores(df_responses: pd.DataFrame) -> pd.DataFrame:
    """Calcula os constructos a partir dos itens do questionário."""
    df = df_responses.copy()

    for item_col in ITEM_COLUMNS:
        if item_col in df.columns:
            df[item_col] = pd.to_numeric(df[item_col], errors="coerce")

    for construct, items in CONSTRUCT_MAP.items():
        existing_items = [item for item in items if item in df.columns]
        if existing_items:
            df[construct] = df[existing_items].mean(axis=1)

    return df


def load_cluster_checkpoints(output_dir: str | Path = RUN_OUTPUT_DIR) -> pd.DataFrame:
    """Lê todos os checkpoints parquet do run e devolve um DataFrame único."""
    output_dir = Path(output_dir)
    files = sorted(output_dir.glob("cluster_responses_checkpoint_*.parquet"))

    if not files:
        raise FileNotFoundError(f"Nenhum checkpoint encontrado em: {output_dir}")

    print("Checkpoints encontrados:", len(files))
    df = pd.concat([pd.read_parquet(file) for file in files], ignore_index=True)
    print("Shape consolidado:", df.shape)

    return df


def count_errors(series: pd.Series) -> int:
    """Conta respostas com erro registrado."""
    return int((series.fillna("").astype(str).str.len() > 0).sum())


def aggregate_cluster_scores(df_scores: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """
    Gera tabelas agregadas para análise final.

    Retorna:
    - agg_cluster_segment: cluster × segmento;
    - agg_macrotema_segment: macrotema × segmento;
    - agg_tema_segment: tema × segmento;
    - agg_cluster: cluster geral;
    - agg_segment: segmento geral.
    """
    df_scores = df_scores.copy()

    construct_cols = [col for col in CONSTRUCT_MAP.keys() if col in df_scores.columns]
    item_cols = [col for col in ITEM_COLUMNS if col in df_scores.columns]
    base_metrics = item_cols + construct_cols

    for col in base_metrics:
        df_scores[col] = pd.to_numeric(df_scores[col], errors="coerce")

    agg_cluster_segment = (
        df_scores
        .groupby(
            [
                "cluster_id",
                "cluster_tema",
                "cluster_macrotema",
                "cluster_motivo",
                "segmento_e_bolhas",
            ],
            dropna=False,
            as_index=False,
        )
        .agg(
            n_respostas=("id_persona", "count"),
            n_agentes=("id_persona", "nunique"),
            **{f"{col}_mean": (col, "mean") for col in base_metrics},
            **{f"{col}_std": (col, "std") for col in base_metrics},
            errors=("error", count_errors),
        )
    )

    agg_macrotema_segment = (
        df_scores
        .groupby(["cluster_macrotema", "segmento_e_bolhas"], dropna=False, as_index=False)
        .agg(
            n_respostas=("id_persona", "count"),
            n_agentes=("id_persona", "nunique"),
            n_clusters=("cluster_id", "nunique"),
            **{f"{col}_mean": (col, "mean") for col in construct_cols},
            **{f"{col}_std": (col, "std") for col in construct_cols},
            errors=("error", count_errors),
        )
    )

    agg_tema_segment = (
        df_scores
        .groupby(["cluster_tema", "cluster_macrotema", "segmento_e_bolhas"], dropna=False, as_index=False)
        .agg(
            n_respostas=("id_persona", "count"),
            n_agentes=("id_persona", "nunique"),
            n_clusters=("cluster_id", "nunique"),
            **{f"{col}_mean": (col, "mean") for col in construct_cols},
            **{f"{col}_std": (col, "std") for col in construct_cols},
            errors=("error", count_errors),
        )
    )

    agg_cluster = (
        df_scores
        .groupby(
            ["cluster_id", "cluster_tema", "cluster_macrotema", "cluster_motivo"],
            dropna=False,
            as_index=False,
        )
        .agg(
            n_respostas=("id_persona", "count"),
            n_agentes=("id_persona", "nunique"),
            n_segmentos=("segmento_e_bolhas", "nunique"),
            **{f"{col}_mean": (col, "mean") for col in construct_cols},
            **{f"{col}_std": (col, "std") for col in construct_cols},
            errors=("error", count_errors),
        )
    )

    agg_segment = (
        df_scores
        .groupby(["segmento_e_bolhas"], dropna=False, as_index=False)
        .agg(
            n_respostas=("id_persona", "count"),
            n_agentes=("id_persona", "nunique"),
            n_clusters=("cluster_id", "nunique"),
            **{f"{col}_mean": (col, "mean") for col in construct_cols},
            **{f"{col}_std": (col, "std") for col in construct_cols},
            errors=("error", count_errors),
        )
    )

    for df_agg in [agg_cluster_segment, agg_macrotema_segment, agg_tema_segment, agg_cluster, agg_segment]:
        numeric_cols = df_agg.select_dtypes(include="number").columns
        df_agg[numeric_cols] = df_agg[numeric_cols].round(3)

    return {
        "agg_cluster_segment": agg_cluster_segment,
        "agg_macrotema_segment": agg_macrotema_segment,
        "agg_tema_segment": agg_tema_segment,
        "agg_cluster": agg_cluster,
        "agg_segment": agg_segment,
    }


def save_complete_and_aggregated_outputs(
    output_dir: str | Path = RUN_OUTPUT_DIR,
) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    """
    Consolida os checkpoints, calcula constructos e salva outputs finais.

    Arquivos principais:
    - synthetic_clusters_complete.parquet
    - synthetic_clusters_complete.csv
    - aggregations/*.parquet
    - aggregations/*.csv
    """
    output_dir = Path(output_dir)
    agg_dir = output_dir / "aggregations"
    agg_dir.mkdir(parents=True, exist_ok=True)

    df_complete = load_cluster_checkpoints(output_dir)
    df_complete = compute_construct_scores(df_complete)

    complete_parquet = output_dir / "synthetic_clusters_complete.parquet"
    complete_csv = output_dir / "synthetic_clusters_complete.csv"

    df_complete.to_parquet(complete_parquet, index=False)
    df_complete.to_csv(complete_csv, index=False, sep=";", encoding="utf-8-sig")

    print("Arquivo completo salvo em:", complete_parquet)
    print("Arquivo completo CSV salvo em:", complete_csv)

    aggregations = aggregate_cluster_scores(df_complete)

    for name, df_agg in aggregations.items():
        parquet_path = agg_dir / f"{name}.parquet"
        csv_path = agg_dir / f"{name}.csv"

        df_agg.to_parquet(parquet_path, index=False)
        df_agg.to_csv(csv_path, index=False, sep=";", encoding="utf-8-sig")

        print(f"Agregado salvo: {parquet_path}")

    return df_complete, aggregations

## 9. Execução final

In [ ]:
if RUN_PIPELINE:
    if CLEAR_EXISTING_OUTPUTS:
        clear_previous_outputs(RUN_OUTPUT_DIR)

    # 1. Carrega ou gera a população de 1000 agentes.
    df_agents_population = load_or_generate_agents(
        n_agents=N_AGENTS_POPULATION,
        seed=RANDOM_STATE,
        force_generate=FORCE_GENERATE_AGENTS,
    )

    # 2. Cria a amostra padrão de 200 agentes.
    df_agents_sample = make_agent_sample(
        df_agents=df_agents_population,
        target=N_AGENTS_SAMPLE,
        stratify_col="segmento_e_bolhas",
        random_state=RANDOM_STATE,
    )

    # 3. Carrega somente o parquet de clusters.
    df_clusters = load_clusters_from_parquet(CLUSTERS_PARQUET)

    # 4. Roda agente sintético × cluster.
    run_synthetic_audience_clusters(
        df_agents=df_agents_sample,
        df_clusters=df_clusters,
        output_dir=RUN_OUTPUT_DIR,
        max_workers=MAX_WORKERS,
        checkpoint_size=CHECKPOINT_SIZE,
        max_retries=MAX_RETRIES,
        retry_sleep=RETRY_SLEEP,
        limit_agents=LIMIT_AGENTS,
        limit_clusters=LIMIT_CLUSTERS,
    )

    # 5. Consolida completo e salva agregados.
    df_clusters_complete, cluster_aggs = save_complete_and_aggregated_outputs(
        output_dir=RUN_OUTPUT_DIR,
    )

    print("Pipeline concluído.")
    print("Completo:", RUN_OUTPUT_DIR / "synthetic_clusters_complete.parquet")
    print("Agregados:", RUN_OUTPUT_DIR / "aggregations")
else:
    print("RUN_PIPELINE está False. Altere para True para executar.")

## 10. Conferência dos arquivos gerados

In [ ]:
print("Pasta do run:", RUN_OUTPUT_DIR)

for path in sorted(RUN_OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(path)